In [43]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [44]:
from IPython.core.display import display, HTML, clear_output
display(HTML('<style>.container { width:90% !important; }</style>'))
display(HTML('<style>.prompt { min-width:10ex !important; }</style>'))
display(HTML('<style>div#notebook { font-size:12px !important; }</style>'))

/tmp/ipykernel_1724/4223674425.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML, clear_output
/tmp/ipykernel_1724/4223674425.py:1: DeprecationWarning: Importing clear_output from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML, clear_output


In [45]:
%%html
<style>
div.output_area pre {
    white-space: pre;
}
</style>

In [46]:
import pyspark
import pyspark.sql.functions as sf

In [47]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("test_app") \
    .getOrCreate()

In [48]:
spark

In [7]:
! pip install kagglehub

In [49]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akalyasubramanian/dataset-for-spark-df-operations")

print("Path to dataset files:", path)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/akalyasubramanian/dataset-for-spark-df-operations/versions/1


In [50]:
! ls /home/ubuntu/.cache/kagglehub/datasets/akalyasubramanian/dataset-for-spark-df-operations/versions/1/

ls: cannot access '/home/ubuntu/.cache/kagglehub/datasets/akalyasubramanian/dataset-for-spark-df-operations/versions/1/': No such file or directory


In [51]:
! hdfs dfs -ls /user/spark_advanced/

Found 2 items
-rw-r--r--   1 root   users   36178879 2026-09-19 09:10 /user/spark_advanced/athletes.csv
drwxr-xr-x   - hadoop users          0 2026-09-19 09:38 /user/spark_advanced/medalists_output


In [11]:
! hdfs dfs -mkdir -p /user/spark_advanced && hadoop fs -put /home/jovyan/.cache/kagglehub/datasets/akalyasubramanian/dataset-for-spark-df-operations/versions/1/athletes.csv /user/spark_advanced/

put: `/user/spark_advanced/athletes.csv': File exists


In [52]:
! ls /opt/conda/share/jupyter/kernels/python3

kernel.json  logo-32x32.png  logo-64x64.png  logo-svg.svg


In [53]:
athletes = spark.read.csv('/user/spark_advanced/athletes.csv', header=True, inferSchema=True)

In [14]:
athletes.show(10)

+---+--------------------+------+---+------+------+--------------+---+-----------+----+------+-----------+-------------+--------------------+-----+
| ID|                Name|Gender|Age|Height|Weight|          Team|NOC|      Games|Year|Season|       City|        Sport|               Event|Medal|
+---+--------------------+------+---+------+------+--------------+---+-----------+----+------+-----------+-------------+--------------------+-----+
|  1|           A Dijiang|     M| 24|   180|    80|         China|CHN|1992 Summer|1992|Summer|  Barcelona|   Basketball|Basketball Men's ...|   NA|
|  2|            A Lamusi|     M| 23|   170|    60|         China|CHN|2012 Summer|2012|Summer|     London|         Judo|Judo Men's Extra-...|   NA|
|  3| Gunnar Nielsen Aaby|     M| 24|    NA|    NA|       Denmark|DEN|1920 Summer|1920|Summer|  Antwerpen|     Football|Football Men's Fo...|   NA|
|  4|Edgar Lindenau Aabye|     M| 34|    NA|    NA|Denmark/Sweden|DEN|1900 Summer|1900|Summer|      Paris|   Tug

In [15]:
type(athletes)

pyspark.sql.dataframe.DataFrame

In [16]:
athletes.dtypes

[('ID', 'int'),
 ('Name', 'string'),
 ('Gender', 'string'),
 ('Age', 'string'),
 ('Height', 'string'),
 ('Weight', 'string'),
 ('Team', 'string'),
 ('NOC', 'string'),
 ('Games', 'string'),
 ('Year', 'int'),
 ('Season', 'string'),
 ('City', 'string'),
 ('Sport', 'string'),
 ('Event', 'string'),
 ('Medal', 'string')]

In [17]:
athletes.columns

['ID',
 'Name',
 'Gender',
 'Age',
 'Height',
 'Weight',
 'Team',
 'NOC',
 'Games',
 'Year',
 'Season',
 'City',
 'Sport',
 'Event',
 'Medal']

## Практика с DataFrame API
В этом разделе разберём основные операции с DataFrame API Spark:
- `select`, `selectExpr` — выбор колонок
- `filter` / `where` — фильтрация строк
- `orderBy` / `sort` — сортировка
- `withColumn`, `withColumnRenamed`, `drop` — управление колонками
- `groupBy` + `agg` — агрегации
- `when` / `otherwise` — условные выражения
- строковые и числовые функции
- `distinct`, `dropDuplicates`
- `join` (базовый обзор)
- `cache` / `persist`
- Window-функции

In [54]:
# Полную схему можно посмотреть через printSchema()
athletes.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Height: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- Team: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Games: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Season: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Sport: string (nullable = true)
 |-- Event: string (nullable = true)
 |-- Medal: string (nullable = true)



In [55]:
# --- select: различные способы выбора колонок ---
# По имени строки
athletes.select("Name", "Age").show(5)

# Через sf.col() — позволяет применять функции к колонке
athletes.select(sf.col("Name"), sf.col("Age") + 1).show(5)

# selectExpr — SQL-выражения прямо в строке
athletes.selectExpr("Name as athlete_name", "Age * 2 as double_age", "upper(Gender) as gender_upper").show(5)

+--------------------+---+
|                Name|Age|
+--------------------+---+
|           A Dijiang| 24|
|            A Lamusi| 23|
| Gunnar Nielsen Aaby| 24|
|Edgar Lindenau Aabye| 34|
|Christine Jacoba ...| 21|
+--------------------+---+
only showing top 5 rows

+--------------------+---------+
|                Name|(Age + 1)|
+--------------------+---------+
|           A Dijiang|     25.0|
|            A Lamusi|     24.0|
| Gunnar Nielsen Aaby|     25.0|
|Edgar Lindenau Aabye|     35.0|
|Christine Jacoba ...|     22.0|
+--------------------+---------+
only showing top 5 rows

+--------------------+----------+------------+
|        athlete_name|double_age|gender_upper|
+--------------------+----------+------------+
|           A Dijiang|      48.0|           M|
|            A Lamusi|      46.0|           M|
| Gunnar Nielsen Aaby|      48.0|           M|
|Edgar Lindenau Aabye|      68.0|           M|
|Christine Jacoba ...|      42.0|           F|
+--------------------+----------+-

### Фильтрация: `filter` / `where`
`filter` и `where` — синонимы. Условия комбинируются через `&` (AND), `|` (OR), `~` (NOT).
Каждое условие нужно брать в скобки!

In [56]:
# Фильтр по одному условию
athletes.filter(sf.col("Sport") == "Basketball").show(5)

+---+--------------------+------+---+------+------+-----+---+-----------+----+------+---------+----------+--------------------+-----+
| ID|                Name|Gender|Age|Height|Weight| Team|NOC|      Games|Year|Season|     City|     Sport|               Event|Medal|
+---+--------------------+------+---+------+------+-----+---+-----------+----+------+---------+----------+--------------------+-----+
|  1|           A Dijiang|     M| 24|   180|    80|China|CHN|1992 Summer|1992|Summer|Barcelona|Basketball|Basketball Men's ...|   NA|
| 69|   Tamara Abalde Daz|     F| 19|   185|    72|Spain|ESP|2008 Summer|2008|Summer|  Beijing|Basketball|Basketball Women'...|   NA|
|124|Youssef Mohamed A...|     M| 31|    NA|    NA|Egypt|EGY|1952 Summer|1952|Summer| Helsinki|Basketball|Basketball Men's ...|   NA|
|136|    Alessandro Abbio|     M| 29|   195|    85|Italy|ITA|2000 Summer|2000|Summer|   Sydney|Basketball|Basketball Men's ...|   NA|
|192|Ahmed El-Sayed Ab...|     M| 25|   189|    85|Egypt|EGY|1

In [61]:
# Несколько условий: AND — баскетболисты из Барселоны
athletes.where(sf.col("Sport") == "Basketball").filter(sf.col("City") == "Barcelona").select("Name", "Team", "Age").show(9)

+--------------------+--------------+---+
|                Name|          Team|Age|
+--------------------+--------------+---+
|           A Dijiang|         China| 24|
|Adriana Aparecida...|        Brazil| 21|
|Judith guila Hern...|          Cuba| 19|
|      Vladan Alanovi|       Croatia| 25|
|Santiago Aldama A...|         Spain| 23|
|Ana Beln lvaro Ba...|         Spain| 23|
|        Armin Andres|       Germany| 33|
|Enrique Andreu Ba...|         Spain| 24|
|       Eva Antalecov|Czechoslovakia| 26|
+--------------------+--------------+---+
only showing top 9 rows



In [58]:
# OR — баскетбол или футбол
athletes.filter(
    (sf.col("Sport") == "Basketball") | (sf.col("Sport") == "Football")
).select("Name", "Sport").show(5)

+-------------------+----------+
|               Name|     Sport|
+-------------------+----------+
|          A Dijiang|Basketball|
|Gunnar Nielsen Aaby|  Football|
| Ann Kristin Aarnes|  Football|
|           Rolf Aas|  Football|
|   Stefan Abadzhiev|  Football|
+-------------------+----------+
only showing top 5 rows



In [62]:
# isin — фильтр по набору значений
athletes.filter(sf.col("Sport").isin("Basketball", "Football", "Judo")).select("Name", "Sport").show(5)

# isNull / isNotNull — проверка на NULL
athletes.filter(sf.col("Medal").isNotNull()).select("Name", "Medal").show(5)

# between — диапазон значений
athletes.filter(sf.col("Age").between(20, 25)).select("Name", "Age").show(5)

+-------------------+----------+
|               Name|     Sport|
+-------------------+----------+
|          A Dijiang|Basketball|
|           A Lamusi|      Judo|
|Gunnar Nielsen Aaby|  Football|
| Ann Kristin Aarnes|  Football|
|           Rolf Aas|  Football|
+-------------------+----------+
only showing top 5 rows

+--------------------+-----+
|                Name|Medal|
+--------------------+-----+
|           A Dijiang|   NA|
|            A Lamusi|   NA|
| Gunnar Nielsen Aaby|   NA|
|Edgar Lindenau Aabye| Gold|
|Christine Jacoba ...|   NA|
+--------------------+-----+
only showing top 5 rows

+--------------------+---+
|                Name|Age|
+--------------------+---+
|           A Dijiang| 24|
|            A Lamusi| 23|
| Gunnar Nielsen Aaby| 24|
|Christine Jacoba ...| 21|
|Christine Jacoba ...| 21|
+--------------------+---+
only showing top 5 rows



In [63]:
# where и filter — одно и то же, можно chaining
athletes \
    .where(sf.col("Sport") == "Basketball") \
    .filter(sf.col("Age") > 30) \
    .select("Name", "Age", "Team") \
    .show(5)

+--------------------+---+------+
|                Name|Age|  Team|
+--------------------+---+------+
|Youssef Mohamed A...| 31| Egypt|
|Adilson de Freita...| 32|Brazil|
|"Adriana ""Adrian...| 33|Brazil|
|"Adriana ""Adrian...| 37|Brazil|
|Arturo Ernesto Agard| 31|Panama|
+--------------------+---+------+
only showing top 5 rows



### Сортировка: `orderBy` / `sort`
`orderBy` и `sort` — синонимы. Используйте `sf.asc()`, `sf.desc()`, `sf.asc_nulls_last()` и т.д.

In [64]:
# Сортировка по возрастанию (одна колонка)
athletes.select("Name", "Age").orderBy("Age").show(5)

# По убыванию
athletes.select("Name", "Age").orderBy(sf.col("Age").desc()).show(5)

# По нескольким колонкам
athletes.select("Name", "Team", "Age").orderBy("Team", sf.col("Age").desc()).show(5)

+--------------------+---+
|                Name|Age|
+--------------------+---+
|  Dimitrios Loundras| 10|
|Megan Olwen Deven...| 11|
|        Liana Vicens| 11|
|   Marcelle Matthews| 11|
|    Luigina Giavotti| 11|
+--------------------+---+
only showing top 5 rows

+--------------------+---+
|                Name|Age|
+--------------------+---+
|John Slaats Verho...| NA|
|     Arsne Kra Konan| NA|
|    William Ellicott| NA|
|         Henry Kraft| NA|
|    Georgi Abadzhiev| NA|
+--------------------+---+
only showing top 5 rows

+----------------+--------------------+---+
|            Name|                Team|Age|
+----------------+--------------------+---+
| Ioannis Georgas|"Large boat, Cent...| NA|
| Ioannis Agrimis|"Large boat, Cent...| NA|
|Mikhail Mouratis|"Large boat, Cent...| NA|
|  Andreas Drivas|"Large boat, Cent...| NA|
| Isidoros Mikhas|"Large boat, Cent...| NA|
+----------------+--------------------+---+
only showing top 5 rows



### Управление колонками: `withColumn`, `withColumnRenamed`, `drop`

In [66]:
athletes.columns

['ID',
 'Name',
 'Gender',
 'Age',
 'Height',
 'Weight',
 'Team',
 'NOC',
 'Games',
 'Year',
 'Season',
 'City',
 'Sport',
 'Event',
 'Medal']

In [68]:
athletes_with_flag.columns

['ID',
 'Name',
 'Gender',
 'Age',
 'Height',
 'Weight',
 'Team',
 'NOC',
 'Games',
 'Year',
 'Season',
 'City',
 'Sport',
 'Event',
 'Medal',
 'is_medalist']

In [67]:
# withColumn — добавить новую колонку или перезаписать существующую
athletes_with_flag = athletes.withColumn("is_medalist", sf.col("Medal").isNotNull())
athletes_with_flag.select("Name", "Medal", "is_medalist").show(5)

# Арифметика: возраст в годах → возраст в днях (приблизительно)
athletes.withColumn("age_in_days", sf.col("Age") * 365).select("Name", "Age", "age_in_days").show(5)

+--------------------+-----+-----------+
|                Name|Medal|is_medalist|
+--------------------+-----+-----------+
|           A Dijiang|   NA|       true|
|            A Lamusi|   NA|       true|
| Gunnar Nielsen Aaby|   NA|       true|
|Edgar Lindenau Aabye| Gold|       true|
|Christine Jacoba ...|   NA|       true|
+--------------------+-----+-----------+
only showing top 5 rows

+--------------------+---+-----------+
|                Name|Age|age_in_days|
+--------------------+---+-----------+
|           A Dijiang| 24|     8760.0|
|            A Lamusi| 23|     8395.0|
| Gunnar Nielsen Aaby| 24|     8760.0|
|Edgar Lindenau Aabye| 34|    12410.0|
|Christine Jacoba ...| 21|     7665.0|
+--------------------+---+-----------+
only showing top 5 rows



In [70]:
# withColumnRenamed — переименовать колонку
athletes_renamed = athletes.withColumnRenamed("Name", "athlete_name")
athletes_renamed.columns

# drop — удалить колонку
athletes.drop("Medal").columns

['ID',
 'Name',
 'Gender',
 'Age',
 'Height',
 'Weight',
 'Team',
 'NOC',
 'Games',
 'Year',
 'Season',
 'City',
 'Sport',
 'Event']

### Условные выражения: `when` / `otherwise`
Аналог CASE WHEN в SQL.

In [71]:
# Категоризация атлетов по возрастным группам
athletes_categorized = athletes.withColumn(
    "age_group",
    sf.when(sf.col("Age") < 20, "junior")
    .when((sf.col("Age") >= 20) & (sf.col("Age") < 30), "adult")
    .when((sf.col("Age") >= 30) & (sf.col("Age") < 40), "veteran")
    .otherwise("senior")
)
athletes_categorized.select("Name", "Age", "age_group").show(10)

+--------------------+---+---------+
|                Name|Age|age_group|
+--------------------+---+---------+
|           A Dijiang| 24|    adult|
|            A Lamusi| 23|    adult|
| Gunnar Nielsen Aaby| 24|    adult|
|Edgar Lindenau Aabye| 34|  veteran|
|Christine Jacoba ...| 21|    adult|
|Christine Jacoba ...| 21|    adult|
|Christine Jacoba ...| 25|    adult|
|Christine Jacoba ...| 25|    adult|
|Christine Jacoba ...| 27|    adult|
|Christine Jacoba ...| 27|    adult|
+--------------------+---+---------+
only showing top 10 rows



### Агрегации: `groupBy` + `agg`
В `agg` можно передать словарь `{"колонка": "функция"}` или функции `sf.count()`, `sf.avg()`, `sf.sum()`, `sf.max()`, `sf.min()`, `sf.countDistinct()`.

In [72]:
# Группировка по команде, количество медалей
athletes.groupBy("Team").agg({"Medal": "count"}).show(10, 0)

+----------------+------------+
|Team            |count(Medal)|
+----------------+------------+
|Chad            |41          |
|Mohawk Indians-2|12          |
|Unified Team-1  |14          |
|Russia          |4922        |
|Paraguay        |135         |
|Elisabeth X     |5           |
|Austria-1       |128         |
|Fortuna         |2           |
|Pampero         |9           |
|United States-10|2           |
+----------------+------------+
only showing top 10 rows



In [73]:
# Несколько агрегаций одновременно через sf-функции
athletes.groupBy("Sport").agg(
    sf.count("*").alias("total_athletes"),
    sf.avg("Age").alias("avg_age"),
    sf.min("Age").alias("min_age"),
    sf.max("Age").alias("max_age"),
    sf.countDistinct("Team").alias("unique_teams")
).orderBy(sf.col("total_athletes").desc()).show(10, 0)

+--------------------+--------------+------------------+-------+-------+------------+
|Sport               |total_athletes|avg_age           |min_age|max_age|unique_teams|
+--------------------+--------------+------------------+-------+-------+------------+
|Athletics           |38624         |25.161223238328844|12     |NA     |231         |
|Gymnastics          |26707         |22.733038232528987|10     |NA     |109         |
|Swimming            |23195         |20.566803405231354|11     |NA     |208         |
|Shooting            |11448         |33.42222640444152 |15     |NA     |152         |
|Cycling             |10859         |24.773990767115215|14     |NA     |150         |
|Fencing             |10735         |28.538822355289422|14     |NA     |101         |
|Rowing              |10595         |25.276293574337075|11     |NA     |181         |
|Cross Country Skiing|9133          |26.006155198944825|15     |NA     |81          |
|Alpine Skiing       |8829          |23.20546223288767

In [ ]:
# countDistinct — количество уникальных значений
athletes.agg(
    sf.countDistinct("Sport").alias("unique_sports"),
    sf.countDistinct("Team").alias("unique_teams"),
    sf.countDistinct("NOC").alias("unique_noc")
).show()

In [ ]:
# Агрегация с переименованием колонки
medals_count = athletes.groupBy("Team").agg(
    sf.count("Medal").alias("medals_count")
)
medals_count.show()

# Средний возраст по видам спорта
average_age_by_sport = athletes.groupBy("Sport").agg(
    sf.avg("Age").alias("average_age")
)
average_age_by_sport.show()

### Строковые функции
Spark предоставляет богатый набор строковых функций: `upper`, `lower`, `concat`, `length`, `substring`, `trim`, `regexp_replace` и др.

In [74]:
# Строковые функции
athletes.select(
    sf.col("Name"),
    sf.upper(sf.col("Name")).alias("name_upper"),
    sf.lower(sf.col("Name")).alias("name_lower"),
    sf.length(sf.col("Name")).alias("name_length"),
    sf.substring(sf.col("Name"), 1, 3).alias("name_first3"),
    sf.concat(sf.col("Name"), sf.lit(" - "), sf.col("Sport")).alias("name_sport")
).show(5, 0)

+------------------------+------------------------+------------------------+-----------+-----------+----------------------------------------+
|Name                    |name_upper              |name_lower              |name_length|name_first3|name_sport                              |
+------------------------+------------------------+------------------------+-----------+-----------+----------------------------------------+
|A Dijiang               |A DIJIANG               |a dijiang               |9          |A D        |A Dijiang - Basketball                  |
|A Lamusi                |A LAMUSI                |a lamusi                |8          |A L        |A Lamusi - Judo                         |
|Gunnar Nielsen Aaby     |GUNNAR NIELSEN AABY     |gunnar nielsen aaby     |19         |Gun        |Gunnar Nielsen Aaby - Football          |
|Edgar Lindenau Aabye    |EDGAR LINDENAU AABYE    |edgar lindenau aabye    |20         |Edg        |Edgar Lindenau Aabye - Tug-Of-War       |
|Chris

In [75]:
# regexp_replace — замена по регулярному выражению
athletes.select(
    sf.col("Name"),
    sf.regexp_replace(sf.col("Name"), "[aeiouAEIOU]", "*").alias("name_no_vowels")
).show(5)

+--------------------+--------------------+
|                Name|      name_no_vowels|
+--------------------+--------------------+
|           A Dijiang|           * D*j**ng|
|            A Lamusi|            * L*m*s*|
| Gunnar Nielsen Aaby| G*nn*r N**ls*n **by|
|Edgar Lindenau Aabye|*dg*r L*nd*n** **by*|
|Christine Jacoba ...|Chr*st*n* J*c*b* ...|
+--------------------+--------------------+
only showing top 5 rows



### `distinct` и `dropDuplicates`
`distinct()` — убирает полностью дублирующиеся строки.
`dropDuplicates(["col1", "col2"])` — убирает дубликаты по указанным колонкам.

In [33]:
# distinct — количество уникальных имён
athletes.select("Name").distinct().count()

# Общее количество записей
athletes.count()

271116

In [76]:
# dropDuplicates — убираем дубликаты по ID атлета
df_unique = athletes.dropDuplicates(["ID"])
df_unique.count()

135571

In [35]:
# Выбор конкретных колонок
selected_columns = athletes.select("Name", "Team", "Medal")
selected_columns.show()

+--------------------+--------------+-----+
|                Name|          Team|Medal|
+--------------------+--------------+-----+
|           A Dijiang|         China|   NA|
|            A Lamusi|         China|   NA|
| Gunnar Nielsen Aaby|       Denmark|   NA|
|Edgar Lindenau Aabye|Denmark/Sweden| Gold|
|Christine Jacoba ...|   Netherlands|   NA|
|Christine Jacoba ...|   Netherlands|   NA|
|Christine Jacoba ...|   Netherlands|   NA|
|Christine Jacoba ...|   Netherlands|   NA|
|Christine Jacoba ...|   Netherlands|   NA|
|Christine Jacoba ...|   Netherlands|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|     Per Knut Aaland| United States|   NA|
|        John Aalberg| United States|   NA|
|        John Aalberg| United St

### Статистика: `describe`, `summary`
Быстрый способ получить сводную статистику по числовым колонкам.

In [ ]:
# describe — count, mean, stddev, min, max
athletes.describe(["Age", "Height", "Weight"]).show()

# summary — квартили + min/max
athletes.select("Age").summary("min", "25%", "50%", "75%", "max").show()

### `join` — базовое соединение
Подробно виды джойнов разбираются в следующем разделе. Здесь — базовый пример.

In [77]:
# Создадим два небольших DataFrame для демонстрации join
data_athletes = [
    (1, "A Dijiang", "M", 24, "China"),
    (2, "A Lamusi", "M", 23, "China"),
    (3, "Gunnar Nielsen Aaby", "M", 24, "Denmark"),
    (4, "Edgar Lindenau Aabye", "M", 34, "Denmark/Sweden"),
    (5, "Christine Jacoba Aaftink", "F", 21, "Netherlands"),
]

columns_athletes = ["ID", "Name", "Gender", "Age", "Team"]
df_athletes = spark.createDataFrame(data_athletes, schema=columns_athletes)

# Второй DataFrame: медали
data_medals = [
    (1, "Gold"),
    (2, "Silver"),
    (3, "Bronze"),
    (4, "Gold"),
    (5, "NA"),
]

columns_medals = ["Athlete_ID", "Medal"]
df_medals = spark.createDataFrame(data_medals, schema=columns_medals)

df_athletes.show()
df_medals.show()

+---+--------------------+------+---+--------------+
| ID|                Name|Gender|Age|          Team|
+---+--------------------+------+---+--------------+
|  1|           A Dijiang|     M| 24|         China|
|  2|            A Lamusi|     M| 23|         China|
|  3| Gunnar Nielsen Aaby|     M| 24|       Denmark|
|  4|Edgar Lindenau Aabye|     M| 34|Denmark/Sweden|
|  5|Christine Jacoba ...|     F| 21|   Netherlands|
+---+--------------------+------+---+--------------+

+----------+------+
|Athlete_ID| Medal|
+----------+------+
|         1|  Gold|
|         2|Silver|
|         3|Bronze|
|         4|  Gold|
|         5|    NA|
+----------+------+



In [78]:
# Inner join по ключу: df_athletes.ID = df_medals.Athlete_ID
df_joined = df_athletes.join(df_medals, df_athletes.ID == df_medals.Athlete_ID, "inner")
df_joined.show()

# left join — все атлеты, даже без медали
df_athletes.join(df_medals, df_athletes.ID == df_medals.Athlete_ID, "left").show()

+---+--------------------+------+---+--------------+----------+------+
| ID|                Name|Gender|Age|          Team|Athlete_ID| Medal|
+---+--------------------+------+---+--------------+----------+------+
|  1|           A Dijiang|     M| 24|         China|         1|  Gold|
|  2|            A Lamusi|     M| 23|         China|         2|Silver|
|  3| Gunnar Nielsen Aaby|     M| 24|       Denmark|         3|Bronze|
|  4|Edgar Lindenau Aabye|     M| 34|Denmark/Sweden|         4|  Gold|
|  5|Christine Jacoba ...|     F| 21|   Netherlands|         5|    NA|
+---+--------------------+------+---+--------------+----------+------+

+---+--------------------+------+---+--------------+----------+------+
| ID|                Name|Gender|Age|          Team|Athlete_ID| Medal|
+---+--------------------+------+---+--------------+----------+------+
|  1|           A Dijiang|     M| 24|         China|         1|  Gold|
|  2|            A Lamusi|     M| 23|         China|         2|Silver|
|  3|

In [79]:
# collect — собрать все данные в список Row на драйвере
# Осторожно с большими DataFrame! Для больших данных используйте show/take/limit
df_athletes.collect()

# take(n) — взять n строк
df_athletes.take(3)

# limit(n) — вернуть новый DataFrame с n строками (лениво)
df_athletes.limit(2).collect()

[Row(ID=1, Name='A Dijiang', Gender='M', Age=24, Team='China'),
 Row(ID=2, Name='A Lamusi', Gender='M', Age=23, Team='China')]

In [87]:
athletes_cached.count()

271116

### `cache` / `persist`
Если DataFrame используется несколько раз, его стоит закешировать, чтобы избежать повторных вычислений.

In [80]:
# cache — сохранить в памяти (default: MEMORY_AND_DISK)
athletes_cached = athletes.cache()

# Теперь повторные действия не будут перечитывать CSV из HDFS
print("Count:", athletes_cached.count())
print("Columns:", athletes_cached.columns)

# persist с конкретным уровнем хранения
from pyspark import StorageLevel
athletes.persist(StorageLevel.MEMORY_ONLY)

# unpersist — освободить память
athletes.unpersist()

Count: 271116
Columns: ['ID', 'Name', 'Gender', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']


DataFrame[ID: int, Name: string, Gender: string, Age: string, Height: string, Weight: string, Team: string, NOC: string, Games: string, Year: int, Season: string, City: string, Sport: string, Event: string, Medal: string]

### Window-функции
Позволяют вычислять значения над "окном" строк — аналог оконных функций в SQL.
`partitionBy` — группировка, `orderBy` — сортировка внутри группы, `rowsBetween` / `rangeBetween` — рамка окна.

In [88]:
from pyspark.sql import Window as W

# Ранг атлета по возрасту внутри каждой команды
window_spec = W.partitionBy("Team").orderBy("Age")

df_with_rank = athletes.withColumn("age_rank", sf.rank().over(window_spec))
df_with_rank.select("Name", "Team", "Age", "age_rank").show(10)

# row_number — сквозная нумерация внутри группы
df_with_rownum = athletes.withColumn(
    "row_num", sf.row_number().over(W.partitionBy("Team").orderBy(sf.col("Age").desc()))
)
df_with_rownum.select("Name", "Team", "Age", "row_num").show(10)

# Накопительная сумма (running total) количества медалей по годам
w_year = W.partitionBy("Team").orderBy("Year").rowsBetween(W.unboundedPreceding, W.currentRow)
athletes.filter(sf.col("Medal") != "NA") \
    .withColumn("running_medals", sf.count("*").over(w_year)) \
    .select("Name", "Team", "Year", "running_medals") \
    .show(15)

+--------------------+-----------+---+--------+
|                Name|       Team|Age|age_rank|
+--------------------+-----------+---+--------+
| Mauricio de la Lama|  Acipactli| 44|       1|
|          Juan Frias|  Acipactli| 46|       2|
|Francisco Eduardo...|  Acipactli| 52|       3|
|      Mohammad Aktar|Afghanistan| 17|       1|
|       Mohammad Asif|Afghanistan| 17|       1|
|     Mohammad Sultan|Afghanistan| 17|       1|
|Faiz Mohammad Kha...|Afghanistan| 18|       4|
|       Friba Razayee|Afghanistan| 18|       4|
|   Robina Muqim Yaar|Afghanistan| 18|       4|
|        Masoud Azizi|Afghanistan| 19|       7|
+--------------------+-----------+---+--------+
only showing top 10 rows

+--------------------+-----------+---+-------+
|                Name|       Team|Age|row_num|
+--------------------+-----------+---+-------+
|Francisco Eduardo...|  Acipactli| 52|      1|
|          Juan Frias|  Acipactli| 46|      2|
| Mauricio de la Lama|  Acipactli| 44|      3|
|   Ahmad Shah Abouw